# Edge-aware SwinIR: Kaggle runner

Runs the whole study with the code in this repository: data preparation → readiness gate → weight sweep → confirmation seeds → analysis.

**Before you start** (right-hand panel of the Kaggle editor):
1. *Accelerator*: **GPU T4 x2** (or P100 / T4).
2. *Internet*: **on** (to clone the code; not needed if you attach the code as a dataset, see cell 1).
3. *Add Input*: dataset **mcparadip/anime-faces-waifu2x**.
4. For long stages use **Save Version → Save & Run All (Commit)** so the run continues after you close the browser. Every stage stops by itself before the 12 h session limit (`--hours`), and re-running a cell resumes where it stopped.

Details: `docs/EXPERIMENTS.md` (protocol) and `docs/PANDUAN_ID.md` (step-by-step, Indonesian).

In [ ]:
# 1. Code. Option A: clone from GitHub (Internet on). Option B: attach the repo as a Kaggle dataset.
import os, pathlib, shutil, subprocess
REPO_URL = "https://github.com/rexxar280903/SwinIR"
BRANCH = "main"
WORK = pathlib.Path("/kaggle/working")
CODE = WORK / "SwinIR"
CODE_DATASET = None   # e.g. "/kaggle/input/edgesr-code" for option B
if not CODE.exists():
    if CODE_DATASET:
        shutil.copytree(CODE_DATASET, CODE)
    else:
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(CODE)], check=True)
os.chdir(CODE)
!git log -1 --format="%h %s" 2>/dev/null || echo "(no git metadata)"
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

In [ ]:
# 2. Paths. The source folder is found automatically (the folder under /kaggle/input with the most images).
def find_source(root="/kaggle/input"):
    best, n_best = None, 0
    for d, _, files in os.walk(root):
        n = sum(f.lower().endswith((".png", ".jpg", ".jpeg", ".webp")) for f in files)
        if n > n_best:
            best, n_best = d, n
    return best, n_best

SRC, n_src = find_source()
DATA = WORK / "data"
RUNS = WORK / "runs"
READY = WORK / "readiness"
GPU_HOURS = 60          # GPU hours for the full ablation, summed over GPUs (2 x T4 -> about 30 notebook hours)
SESSION_HOURS = 11.5    # stop starting new runs before the 12 h session limit
IMPORT = []             # previous session outputs, e.g. ["/kaggle/input/edgesr-run-v1/runs"]
print(f"source: {SRC} ({n_src} images)")

In [ ]:
# 3. Build the dataset once (about 2-4 minutes): train/val/test = 80/10/10, duplicates grouped.
if not (DATA / "manifest.csv").exists():
    !python scripts/prepare_data.py --src "{SRC}" --out {DATA}
!cat {DATA}/dataset_info.json

In [ ]:
# 4. (Optional) Reproduce the audit of the original notebook on real images (docs/AUDIT.md).
!python scripts/audit_legacy.py --data {DATA} --n 256 | tee {WORK}/audit_legacy_real.json

In [ ]:
# 5. READINESS GATE (about 5-10 minutes). Read the report before going on.
!python scripts/readiness_check.py --data {DATA} --out {READY} --gpu-hours {GPU_HOURS} --require-gpu
from IPython.display import Markdown, display
display(Markdown((READY / "readiness_report.md").read_text()))

### Stop and read the report
* **FAIL** anywhere → fix it first; do not start the sweep.
* **R6.1 over budget** → set `total_iters` in `configs/base.json` to the suggested value (commit it to GitHub, re-clone), then re-run cell 5.
* **R7.1 protocol lock** → download `readiness/protocol_lock.json` (Output tab), put it in `configs/`, commit and push. From then on the gate checks that nothing in the protocol changes.
* Only then continue with cell 6.

In [ ]:
# 6. Helper: run one stage on 1 or 2 GPUs (2 GPUs -> two shards in parallel).
import sys, torch

def run_stage(stage, extra=""):
    imp = ("--import " + " ".join(IMPORT)) if IMPORT else ""
    base = (f"{sys.executable} scripts/run_ablation.py --data {DATA} --runs {RUNS} "
            f"--stage {stage} --hours {SESSION_HOURS} {imp} {extra}")
    n_gpu = torch.cuda.device_count()
    if n_gpu >= 2 and stage in ("sweep", "extend", "confirm"):
        procs = [subprocess.Popen(f"CUDA_VISIBLE_DEVICES={g} {base} --shard {g + 1}/2 > {WORK}/{stage}_gpu{g}.log 2>&1",
                                  shell=True) for g in range(2)]
        codes = [p.wait() for p in procs]
        for g in range(2):
            print(f"--- GPU {g} (tail) ---")
            print("".join(open(f"{WORK}/{stage}_gpu{g}.log").readlines()[-15:]))
        return codes
    return subprocess.run(base, shell=True).returncode

!python scripts/run_ablation.py --data {DATA} --runs {RUNS} --stage all --list

In [ ]:
# 7. Stage 1: L1 + static/adaptive weight sweep (seed 0, 11 runs), then the bicubic baseline.
run_stage("sweep")
run_stage("bicubic")
!python scripts/run_ablation.py --data {DATA} --runs {RUNS} --stage all --list

In [ ]:
# 8. Stage 2: weights chosen from the sweep's VALIDATION logs. If a chosen weight sits at the
#    end of the grid, the pre-registered extension adds 2 weights first (seed 0). Then seeds 1-3.
run_stage("extend")
run_stage("confirm")
!head -c 3000 {RUNS}/selection.json

In [ ]:
# 9. Analysis: tables, pre-registered hypothesis tests, figures -> runs/analysis/RESULTS.md
!python scripts/analyze.py --runs {RUNS} --data {DATA}
display(Markdown((RUNS / "analysis" / "RESULTS.md").read_text()))

In [ ]:
# 10. Pack the results for download (checkpoints for resuming are not included).
!cd {WORK} && zip -qr results.zip runs readiness audit_legacy_real.json -x "*/last.pt" && ls -lh results.zip

### Continuing in a new session
1. In the finished version's page: *Output* → **New Dataset** (or use the version as input directly) so the `runs/` folder is available under `/kaggle/input/...`.
2. Add it as input to this notebook, set `IMPORT = ["/kaggle/input/<name>/runs"]` in cell 2, and run cells 1-3, 6 and the stage you were in. Finished runs are copied and skipped; nothing is trained twice.